# 00 — Set up the store data

Every other notebook in this topic loads a **trained GMS store** from
`data/gms_annual_report_store/`. That store is a build artifact: it is
**not committed to the repo and not shipped in the package** (it is
git-ignored, like all trained weights and GMS stores). This notebook is the
one place that regenerates it, from the committed corpus
(`data/annual_report.md`) using the scripts in `code/scripts/`.

**What you need**

- The licensed **`knowlytix`** substrate: `pip install knowlytix` and a
  developer license from https://knowlytix.ai/signup/ (key at
  `~/.knowlytix/license.key`). If you use a source checkout instead, set
  `KNOWLYTIX_SRC` to it. Without the substrate, the stores cannot be built.
- `torch`. A GPU is optional for the base store (the corpus is small) but
  strongly recommended for the optional Qwen stages below.

**Two tiers**

1. **Base store** (required by most chapters) — the GEODE-corrected graph +
   cap geometry + ENM. Fast; no LLM.
2. **Full pipeline** (optional; needed only by the advanced chapters —
   enrichment, fine-tuned encoders, calibrated gates). Heavier: loads Qwen.

Each stage is **idempotent** — it is skipped when its output already exists,
so you can re-run this notebook freely.

## 1. Bootstrap: locate `knowlytix` and this repo

In [ ]:
# Self-contained bootstrap (mirrors book_kit.resolve_knowlytix so this cell
# works before book_kit — which imports knowlytix — can be imported).
import importlib.util, os, sys

REPO = os.environ.get('GMS_RAG_TUTORIAL')
if not REPO:
    # notebooks/ is a sibling of code/; resolve code/ from the cwd.
    cwd = os.getcwd()
    REPO = os.path.join(os.path.dirname(cwd), 'code') if os.path.basename(cwd) == 'notebooks' else cwd
os.environ['GMS_RAG_TUTORIAL'] = REPO
for p in (REPO, os.path.join(REPO, 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)

def resolve_knowlytix():
    if importlib.util.find_spec('knowlytix') is not None:
        return 'already importable'
    for c in [os.environ.get('KNOWLYTIX_SRC'),
              os.path.expanduser('~/source/GMS-knowlytix'),
              os.path.expanduser('~/GMS-knowlytix'),
              os.path.normpath(os.path.join(REPO, '..', '..', 'GMS-knowlytix'))]:
        if c and os.path.isdir(os.path.join(c, 'knowlytix')):
            sys.path.insert(0, c); os.environ['KNOWLYTIX_SRC'] = c
            return c
    raise ModuleNotFoundError(
        'knowlytix not found. Install it (`pip install knowlytix`, licensed) or, '
        "for a source checkout, set os.environ['KNOWLYTIX_SRC']='/path/to/GMS-knowlytix', "
        'then restart the kernel.')

print('knowlytix:', resolve_knowlytix())
import torch
print('torch:', torch.__version__, '| device:',
      'cuda' if torch.cuda.is_available() else 'cpu')
print('repo:', REPO)

## 2. Idempotent stage runner

In [ ]:
import subprocess, sys, os

def stage(title, script, args=(), produces=()):
    """Run scripts/<script> unless every path in `produces` already exists.
    Streams the script's output; raises on non-zero exit."""
    produces = list(produces)
    if produces and all(os.path.exists(os.path.join(REPO, p)) for p in produces):
        print(f'\u2713 {title}: already built \u2014 skipping')
        return
    cmd = [sys.executable, os.path.join(REPO, 'scripts', script), *map(str, args)]
    print(f'\u25b6 {title}: python scripts/{script} ' + ' '.join(map(str, args)))
    r = subprocess.run(cmd, cwd=REPO, env=os.environ.copy())
    if r.returncode:
        raise RuntimeError(f'{script} failed (exit {r.returncode})')
    print(f'\u2713 {title}: done')

## 3. Stage 1 — base store (required)

GEODE self-corrects `data/annual_report.md` into a trained store and writes
the ground-truth facts sheet. Skipped if the store already exists.

In [ ]:
stage('Build base store', 'build_store.py',
      produces=['data/gms_annual_report_store/model.pt',
                'data/corpus_facts.md'])

## 4. Verify the base store loads

In [ ]:
from book_kit import load_store

store = load_store()
print('entities:', store.adapter.num_entities,
      '| relations:', store.adapter.num_relations)

## 5. Stage 2–4 — full pipeline (optional; loads Qwen on GPU)

The advanced chapters (calibration, pluggable LLMs, the capstone accept
gate) also need the DoE-enriched training data, the fine-tuned v/u encoders,
and the calibrated gates. These are **heavier** — enrichment and the accept
gate load Qwen — so they are opt-in. Set `RUN_FULL = True` to build them.
Each stage is idempotent.

In [ ]:
RUN_FULL = False  # set True to build enrichment + tuned encoders + gates (needs Qwen/GPU)

if RUN_FULL:
    stage('Enrich (DoE over the store; loads Qwen)', 'enrich_data.py',
          args=['--per-seed', 6],
          produces=['data/enrichment/embedding_sft.jsonl'])
    stage('Fine-tune encoders + relevance gate', 'finetune_encoders.py',
          produces=['data/gms_annual_report_store/tuned_encoder'])
    stage('Calibrate groundedness + relevance gates', 'calibrate_gates.py',
          produces=['data/gms_annual_report_store/calibration.json'])
    stage('Calibrate accept/abstain gate (loads Qwen)', 'calibrate_accept_gate.py',
          produces=['data/gms_annual_report_store/rag_gate_calibration.json'])
    print('\nfull pipeline complete.')
else:
    print('RUN_FULL is False \u2014 base store only. '
          'Set it True for the advanced chapters.')

## Done

The store data now lives under `data/` (git-ignored, never packaged). Open
the chapter notebooks (`01_`, `02_`, …) — they load it through
`book_kit.load_store()`. Re-run this notebook any time to rebuild; existing
artifacts are skipped.